Current format check

In [14]:
from core.file_manager import preprocess_file_manager

from settings.main_settings import test_settings

import numpy as np

In [15]:
settings = test_settings().get_setting_dictionary()
preprocessed_steps = settings['preprocessed_steps']
preprocessing_steps_list = settings['preprocessing_steps_list']
channels = settings['channels']
original_data_folder = settings['original_data_folder']
target_spacing = settings['target_spacing']
crop_size = settings['crop_size']

patients = ['3322','001','003']


file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)

In [16]:
last_step = preprocessed_steps[preprocessing_steps_list[-1][0]]['end']

In [17]:
patient = file_manager.get_file_names()[0]
patients = file_manager.get_file_names()
patient_data = file_manager.load_file_pickle(last_step,'3322')

In [18]:
patient_data

{'adc': array([[[ 451,  216,  248, ...,  260,  188,  277],
         [ 362,  256,  279, ...,  210,  273,  387],
         [ 286,  289,  293, ...,  219,  403,  477],
         ...,
         [ 842,  738,  196, ...,  882,  875, 1002],
         [ 907,  772,  258, ...,  773,  866,  989],
         [ 881,  724,  308, ...,  693,  859,  969]],
 
        [[ 264,  187,  205, ...,  207,  165,  323],
         [ 188,  211,  218, ...,  177,  231,  389],
         [ 140,  235,  228, ...,  207,  353,  451],
         ...,
         [ 739,  857,  221, ...,  862,  855, 1041],
         [ 811,  887,  271, ...,  798,  882, 1050],
         [ 797,  839,  310, ...,  752,  898, 1039]],
 
        [[ 137,  184,  153, ...,  171,  179,  368],
         [  84,  174,  135, ...,  169,  234,  377],
         [  67,  177,  137, ...,  232,  349,  398],
         ...,
         [ 538,  857,  222, ...,  837,  813, 1064],
         [ 584,  899,  250, ...,  827,  874, 1096],
         [ 573,  873,  291, ...,  820,  916, 1099]],
 
      

In [ ]:
dataset = {}


for patient in patients:
    patient_data = file_manager.load_file_pickle(last_step,patient)
    patiend_data_shard = {
        'adc': patient_data['adc'],
        'dwi': patient_data['dwi'],
        't2': patient_data['t2'],
        'anatomy': patient_data['anatomy'],
        'lesion': patient_data['lesion'],

        'label': int(patient_data['lesion'].any())
    }
    dataset[patient] = patiend_data_shard

Output that i need ... 

In [20]:
import os
import pickle
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage


def prepare_dataset(dataset, pp_dir):
    """
    dataset =
    {
        pid: {
            "adc": ndarray or sitk.Image,
            "dwi": ndarray or sitk.Image,
            "t2": ndarray or sitk.Image,
            "anatomy": ndarray or sitk.Image,
            "lesion": ndarray or sitk.Image,
            "label": int
        }
    }
    """

    os.makedirs(pp_dir, exist_ok=True)

    info_rows = []

    for pid, sample in dataset.items():

        def to_numpy(img):
            """Convert SimpleITK.Image -> numpy (x,y,z)."""
            if isinstance(img, sitk.Image):
                arr = sitk.GetArrayFromImage(img)      # (z,y,x)
                arr = np.transpose(arr, (2, 1, 0))     # -> (x,y,z)
                spacing = img.GetSpacing()
            else:
                arr = img
                spacing = None
            return arr, spacing

        adc, spacing = to_numpy(sample["adc"])
        dwi, _ = to_numpy(sample["dwi"])
        t2, _ = to_numpy(sample["t2"])
        anatomy, _ = to_numpy(sample["anatomy"])
        lesion, _ = to_numpy(sample["lesion"])

        # ---------------------------
        # Build 4-channel image
        # ---------------------------

        img = np.stack([
            adc.astype(np.float32),
            dwi.astype(np.float32),
            t2.astype(np.float32),
            anatomy.astype(np.float32),   # prostate mask as 4th channel
        ], axis=-1)

        # ---------------------------
        # Lesion instance mask
        # ---------------------------

        lesion = lesion.astype(np.uint16)

        # If binary, convert to connected-component instances
        if np.unique(lesion).tolist() in ([0], [0, 1], [1]):
            lesion, _ = ndimage.label(lesion > 0)

        seg = lesion[..., None].astype(np.uint16)

        # ---------------------------
        # Save arrays
        # ---------------------------

        np.save(os.path.join(pp_dir, f"{pid}_img.npy"), img)
        np.save(os.path.join(pp_dir, f"{pid}_rois.npy"), seg)

        # ---------------------------
        # Metadata
        # ---------------------------

        fg_slices = np.where(lesion.sum(axis=(0, 1)) > 0)[0].tolist()

        meta = {
            "pid": pid,
            "class_target": [sample["label"]],  # MDT expects a list
            "spacing": spacing,
            "fg_slices": fg_slices,
        }

        with open(os.path.join(pp_dir, f"{pid}_meta_info.pickle"), "wb") as f:
            pickle.dump(meta, f)

        info_rows.append(meta)

    # Build info_df.pickle
    df = pd.DataFrame(info_rows)
    df.to_pickle(os.path.join(pp_dir, "info_df.pickle"))

    print(f"Prepared {len(info_rows)} patients.")

In [21]:
prepare_dataset(dataset, '/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed/pp_dataset')

Prepared 3 patients.
